# Loading Kaggle Malware Datasets

This notebook demonstrates how to load and prepare the two Kaggle datasets:
1. **Malware Analysis Datasets: Raw PE as Image**
2. **Malware Analysis Datasets: API Call Sequences**

## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from utils.kaggle_loader import KaggleMalwareLoader, prepare_dual_branch_data

## 2. Download Datasets from Kaggle

First, you need to download the datasets from Kaggle. You can do this in two ways:

### Option 1: Manual Download
1. Go to the dataset pages:
   - [PE Images](https://www.kaggle.com/datasets/ang3loliveira/malware-analysis-datasets-raw-pe-as-image)
   - [API Sequences](https://www.kaggle.com/datasets/ang3loliveira/malware-analysis-datasets-api-call-sequences)
2. Click "Download" and extract to `data/raw/`

### Option 2: Using Kaggle API
```bash
# Install Kaggle API
pip install kaggle

# Set up your Kaggle API credentials (see https://www.kaggle.com/docs/api)
# Download datasets
kaggle datasets download -d ang3loliveira/malware-analysis-datasets-raw-pe-as-image -p data/raw/pe_images
kaggle datasets download -d ang3loliveira/malware-analysis-datasets-api-call-sequences -p data/raw/api_sequences

# Unzip
unzip data/raw/pe_images/*.zip -d data/raw/pe_images/
unzip data/raw/api_sequences/*.zip -d data/raw/api_sequences/
```

## 3. Set Dataset Paths

Update these paths to match where you extracted the datasets:

In [ ]:
# Update these paths based on where you extracted the datasets
PE_IMAGE_PATH = '../data/raw/pe_images'
API_SEQUENCE_PATH = '../data/raw/api_sequences'

# Check if paths exist
if not Path(PE_IMAGE_PATH).exists():
    print(f"⚠️ PE Image path not found: {PE_IMAGE_PATH}")
    print("Please download and extract the dataset first!")
else:
    print(f"✓ PE Image path found: {PE_IMAGE_PATH}")

if not Path(API_SEQUENCE_PATH).exists():
    print(f"⚠️ API Sequence path not found: {API_SEQUENCE_PATH}")
    print("Please download and extract the dataset first!")
else:
    print(f"✓ API Sequence path found: {API_SEQUENCE_PATH}")

## 4. Initialize Loader

In [ ]:
loader = KaggleMalwareLoader(PE_IMAGE_PATH, API_SEQUENCE_PATH)

## 5. Load PE Images

In [ ]:
pe_images, pe_labels, pe_filenames = loader.load_pe_images(
    target_size=(256, 256),
    normalize=True
)

print(f"\nPE Images shape: {pe_images.shape}")
print(f"PE Labels shape: {pe_labels.shape}")
print(f"Number of files: {len(pe_filenames)}")

## 6. Visualize Sample PE Images

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(10):
    axes[i].imshow(pe_images[i][0], cmap='gray')
    label_name = 'Malware' if pe_labels[i] == 1 else 'Benign'
    axes[i].set_title(f'{label_name}\n{pe_filenames[i][:20]}...')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 7. Load API Call Sequences

In [ ]:
api_sequences, api_labels, api_filenames, vocab = loader.load_api_sequences(
    max_length=1000
)

print(f"\nAPI Sequences shape: {api_sequences.shape}")
print(f"API Labels shape: {api_labels.shape}")
print(f"Vocabulary size: {len(vocab)}")
print(f"\nSample vocabulary (first 20 items):")
for i, (token, idx) in enumerate(list(vocab.items())[:20]):
    print(f"  {token}: {idx}")

## 8. Analyze API Sequences

In [ ]:
# Analyze sequence lengths (before padding)
seq_lengths = []
for seq in api_sequences:
    # Count non-zero elements (excluding padding)
    length = np.count_nonzero(seq)
    seq_lengths.append(length)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(seq_lengths, bins=50, edgecolor='black')
plt.xlabel('Sequence Length')
plt.ylabel('Frequency')
plt.title('Distribution of API Sequence Lengths')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(seq_lengths)
plt.ylabel('Sequence Length')
plt.title('Sequence Length Statistics')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean sequence length: {np.mean(seq_lengths):.2f}")
print(f"Median sequence length: {np.median(seq_lengths):.2f}")
print(f"Max sequence length: {np.max(seq_lengths)}")
print(f"Min sequence length: {np.min(seq_lengths)}")

## 9. Prepare Complete Dataset for Training

In [ ]:
# Prepare and save processed data
data = prepare_dual_branch_data(
    pe_image_path=PE_IMAGE_PATH,
    api_sequence_path=API_SEQUENCE_PATH,
    train_split=0.7,
    val_split=0.15,
    pe_size=(256, 256),
    seq_length=1000,
    save_path='../data/processed'
)

## 10. Verify Processed Data

In [ ]:
print("Training Set:")
print(f"  CNN input shape: {data['train_cnn'].shape}")
print(f"  LSTM input shape: {data['train_lstm'].shape}")
print(f"  Labels shape: {data['train_labels'].shape}")
print(f"  Label distribution: {np.bincount(data['train_labels'])}")

print("\nValidation Set:")
print(f"  CNN input shape: {data['val_cnn'].shape}")
print(f"  LSTM input shape: {data['val_lstm'].shape}")
print(f"  Labels shape: {data['val_labels'].shape}")
print(f"  Label distribution: {np.bincount(data['val_labels'])}")

print("\nTest Set:")
print(f"  CNN input shape: {data['test_cnn'].shape}")
print(f"  LSTM input shape: {data['test_lstm'].shape}")
print(f"  Labels shape: {data['test_labels'].shape}")
print(f"  Label distribution: {np.bincount(data['test_labels'])}")

print(f"\nVocabulary size: {data['vocab_size']}")

## 11. Load Processed Data (For Future Use)

In [ ]:
# You can load the processed data later without reprocessing
import pickle

def load_processed_data(data_path='../data/processed'):
    """Load preprocessed data from disk"""
    data_path = Path(data_path)
    
    data = {
        'train_cnn': np.load(data_path / 'train_cnn.npy'),
        'train_lstm': np.load(data_path / 'train_lstm.npy'),
        'train_labels': np.load(data_path / 'train_labels.npy'),
        
        'val_cnn': np.load(data_path / 'val_cnn.npy'),
        'val_lstm': np.load(data_path / 'val_lstm.npy'),
        'val_labels': np.load(data_path / 'val_labels.npy'),
        
        'test_cnn': np.load(data_path / 'test_cnn.npy'),
        'test_lstm': np.load(data_path / 'test_lstm.npy'),
        'test_labels': np.load(data_path / 'test_labels.npy'),
    }
    
    with open(data_path / 'vocab.pkl', 'rb') as f:
        data['vocab'] = pickle.load(f)
    
    data['vocab_size'] = len(data['vocab'])
    
    return data

# Example usage:
# loaded_data = load_processed_data()

## Next Steps

Now that you have loaded and processed the data:

1. ✅ Data is saved in `data/processed/`
2. 📊 You can proceed to train the dual-branch model
3. 🔬 Use the `03_train_model.ipynb` notebook for training
4. 🎯 The vocabulary size will be used to configure the LSTM input dimension